# **Leukemia Subtype Classification via Gene Expression Profiles Using BioMedGPT-LM-7B**

In [ ]:

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("GPU belum aktif. Periksa pengaturan Runtime.")


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import zipfile
import os

ZIP_PATH = "/content/archive.zip"
DATA_DIR = "/content/gene_expression_data"

os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset berhasil diekstrak!")
print("\nDaftar file:")

for file in os.listdir(DATA_DIR):
    print(file)


In [ ]:

import pandas as pd
import os

DATA_DIR = "/content/gene_expression_data"

train = pd.read_csv(
    os.path.join(DATA_DIR, "data_set_ALL_AML_train.csv")
)

test = pd.read_csv(
    os.path.join(DATA_DIR, "data_set_ALL_AML_independent.csv")
)

labels = pd.read_csv(
    os.path.join(DATA_DIR, "actual.csv")
)

print("Training Shape:", train.shape)
print("Testing Shape:", test.shape)
print("Labels Shape:", labels.shape)

display(train.head())


# **Part 1 - Business Problem**

**🇺🇸 English**

Acute leukemia diagnosis traditionally relies on morphological and cytogenetic analyses, which are time-consuming, costly, and highly dependent on specialist expertise. Misclassification between **Acute Lymphoblastic Leukemia (ALL)** and **Acute Myeloid Leukemia (AML)** directly impacts treatment protocols, since each subtype responds to fundamentally different therapeutic approaches. A wrong diagnosis can lead to ineffective treatment, disease progression, and increased patient mortality.

With the advent of DNA microarray technology, it became possible to capture the gene expression profile of thousands of genes simultaneously from a tumor sample. However, interpreting this high-dimensional data — with **7,129 gene features per patient** and only **72 samples** — poses a significant challenge for conventional machine learning models due to the extreme **curse of dimensionality** and the **small sample size**.

This project proposes leveraging **BioMedGPT-LM-7B**, a large language model pre-trained on biomedical literature and molecular data, to encode, interpret, and classify gene expression profiles from the Golub et al. dataset. The goal is to build a robust, generalizable pipeline that:

- Accurately distinguishes ALL from AML patients using gene expression signatures
- Demonstrates whether biomedical LLMs can serve as feature extractors or zero/few-shot classifiers in genomic tasks
- Provides interpretable outputs aligned with clinical decision-making needs

**Business impact:** Faster, more accurate, and cost-effective leukemia subtype classification to support oncologists in early-stage diagnosis and personalized treatment planning.

**🇧🇷 Português**

O diagnóstico tradicional de leucemia aguda depende de análises morfológicas e citogenéticas que são demoradas, onerosas e altamente dependentes de especialistas. A classificação incorreta entre **Leucemia Linfoide Aguda (LLA/ALL)** e **Leucemia Mieloide Aguda (LMA/AML)** impacta diretamente os protocolos de tratamento, pois cada subtipo responde a abordagens terapêuticas fundamentalmente distintas. Um diagnóstico equivocado pode resultar em tratamento ineficaz, progressão da doença e aumento da mortalidade dos pacientes.

Com o advento da tecnologia de microarray de DNA, tornou-se possível capturar o perfil de expressão gênica de milhares de genes simultaneamente a partir de uma amostra tumoral. No entanto, interpretar esses dados de alta dimensionalidade — com **7.129 features gênicas por paciente** e apenas **72 amostras** — representa um desafio significativo para modelos tradicionais de machine learning, pela extrema **maldição da dimensionalidade** e pelo **tamanho reduzido da amostra**.

Este projeto propõe utilizar o **BioMedGPT-LM-7B**, um modelo de linguagem de grande porte pré-treinado em literatura biomédica e dados moleculares, para codificar, interpretar e classificar perfis de expressão gênica do dataset de Golub et al. O objetivo é construir um pipeline robusto e generalizável que:

- Distinga com precisão pacientes com ALL e AML por meio de assinaturas de expressão gênica
- Demonstre se LLMs biomédicos podem atuar como extratores de features ou classificadores zero/few-shot em tarefas genômicas
- Forneça saídas interpretáveis alinhadas às necessidades de tomada de decisão clínica

**Impacto de negócio:** Classificação de subtipos de leucemia mais rápida, precisa e custo-eficiente para apoiar oncologistas no diagnóstico precoce e no planejamento de tratamento personalizado.

**Resumo Técnico do Escopo**

| Item | Detalhe |
|---|---|
| **Dataset** | Golub et al. — 72 pacientes, 7.129 genes |
| **Target** | ALL vs AML (binário) |
| **Train set** | 38 amostras (`data_set_ALL_AML_train.csv`) |
| **Test set** | 34 amostras (`data_set_ALL_AML_independent.csv`) |
| **Labels** | `actual.csv` (patient + cancer type) |
| **Modelo LLM** | BioMedGPT-LM-7B |
| **Tarefa** | Classificação supervisionada + interpretabilidade |
| **Linguagem** | Python |


# **Part 2 - API Kaggle**

In [ ]:
#from google.colab import files   # (se estiver no Colab)
#import os, json, zipfile

#files.upload()

In [ ]:
#def setup_kaggle_credentials():
    #"""
    #Upload kaggle.json and configure credentials
    #in the default directory expected by the Kaggle CLI.
    #"""
    #files.upload()
    #os.makedirs('/root/.kaggle', exist_ok=True)
    #os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    #os.chmod('/root/.kaggle/kaggle.json', 0o600)

#def download_dataset(dataset_slug: str, dest_dir: str):
    #"""
    #Download a Kaggle dataset via CLI.

    #Args:
        #dataset_slug: Dataset identifier in 'owner/dataset-name' format.
        #dest_dir: Destination directory for the downloaded files.
    #"""
    #os.makedirs(dest_dir, exist_ok=True)
    #os.system(f'kaggle datasets download -d {dataset_slug} --path {dest_dir}')


#def extract_dataset(dest_dir: str, dataset_name: str):
    #"""
    #Extract the zip file downloaded by Kaggle into the destination directory.
    #Args:
        #dest_dir: Directory where the zip file was saved.
        #dataset_name: Dataset name used to locate the zip file.
    #"""
    #zip_path = os.path.join(dest_dir, f'{dataset_name}.zip')

    #i#f not os.path.exists(zip_path):
        #print(f'Zip file not found at: {zip_path}')
        #print('Available files:')
        #for f in os.listdir(dest_dir):
            #print(f'  {f}')
        #return

    #with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        #zip_ref.extractall(dest_dir)
    #print(f'Dataset successfully extracted to: {dest_dir}')

In [ ]:
# KAGGLE CREDENTIALS SETUP
#setup_kaggle_credentials()

In [ ]:
# DATASET DOWNLOAD
#DATASET_SLUG = 'crawford/gene-expression'
#DEST_DIR     = '/content/gene_expression_data'

#download_dataset(DATASET_SLUG, DEST_DIR)

In [ ]:
# ZIP EXTRACTION
#extract_dataset(DEST_DIR, 'gene-expression')

# FILE VERIFICATION
#print('\nAvailable files:')
#for f in os.listdir(DEST_DIR):
    #print(f'  {f}')

# **Part 3 - Database**

In [ ]:
!pip install -q bitsandbytes
!pip install -q transformers==4.41.2
!pip install -q accelerate==0.29.3
!pip install -q xgboost scikit-learn pandas matplotlib
!pip install -q seaborn wordcloud

In [ ]:

import transformers
import torch
import bitsandbytes

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("BitsAndBytes:", bitsandbytes.__version__)


In [ ]:
"""
Environment setup and dependency validation for LLM-based classification pipeline.

This module consolidates all required imports, configures runtime warnings,
and validates the availability of CUDA hardware acceleration.
"""

# STANDARD LIBRARY IMPORTS

import os
import warnings

# THIRD-PARTY NUMERICAL AND DATA IMPORTS

import numpy as np
import pandas as pd

# VISUALISATION IMPORTS

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# PYTORCH IMPORTS

import torch
import torch.nn as nn
import torch.optim as optim

# SCIKIT-LEARN IMPORTS

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# HUGGING FACE TRANSFORMERS IMPORTS

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    GenerationConfig,
    pipeline,
)

# RUNTIME CONFIGURATION

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# HARDWARE VALIDATION

print(f"torch   : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Part 4 - GPU

In [ ]:
"""
Verify installed package versions and available GPU devices.

This module checks the versions of core dependencies and enumerates
all CUDA-capable devices with their respective VRAM capacity.
"""

# Dependency version check

import numpy as np
import torch
import transformers
import accelerate

print(f"numpy        : {np.__version__}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"accelerate   : {accelerate.__version__}")

# CUDA availability and device enumeration

print(f"CUDA ok      : {torch.cuda.is_available()}")
print(f"GPUs         : {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total = props.total_memory / 1024 ** 3
    print(f"  GPU {i} : {props.name} | VRAM: {total:.1f} GB")

In [ ]:

import torch
from transformers import BitsAndBytesConfig

# Dataset configuration
KAGGLE_INPUT = "/content/gene_expression_data"

TRAIN_PATH = f"{KAGGLE_INPUT}/data_set_ALL_AML_train.csv"
TEST_PATH = f"{KAGGLE_INPUT}/data_set_ALL_AML_independent.csv"
LABELS_PATH = f"{KAGGLE_INPUT}/actual.csv"

# Model configuration
MODEL_NAME = "PharMolix/BioMedGPT-LM-7B"

K_GENES = 10
N_SHOTS = 4
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# 4-bit Quantization Configuration

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Quantization: 4-bit NF4")

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Training:", TRAIN_PATH)
print("Testing:", TEST_PATH)
print("Labels:", LABELS_PATH)

# Part 5 - Database

In [ ]:
"""
Loading and transposing CSVs from the Golub dataset.

The original CSV stores genes as rows and patients as columns.
Columns containing 'call' are filtered using safe integer conversion,
because columns such as 'call.1' and 'call.2' are not captured by
an endswith('call') filter and raise ValueError when cast to int.
"""

# Load raw CSV files
train_df  = pd.read_csv(TRAIN_PATH)
test_df   = pd.read_csv(TEST_PATH)
labels_df = pd.read_csv(LABELS_PATH)

# Remove non-feature annotation columns
non_feature_cols   = ["Gene Description", "Gene Accession Number"]
train_patient_cols = [c for c in train_df.columns if c not in non_feature_cols]
test_patient_cols  = [c for c in test_df.columns  if c not in non_feature_cols]

# Preserve gene accession names for later reference
gene_names = train_df["Gene Accession Number"].tolist()

# Transpose so that rows become patients and columns become genes
df_train = train_df[train_patient_cols]
df_test  = test_df[test_patient_cols]
X_train = df_train.T
X_test  = df_test.T

In [ ]:
def is_numeric_col(col):
    """
    Check whether a column name can be cast to an integer.

    Used to filter out qualitative annotation columns such as
    'call', 'call.1', and 'call.2' that do not represent
    gene expression values.

    Args:
        col (str): Column name to evaluate.

    Returns:
        bool: True if the column name represents a numeric patient ID.
    """
    try:
        int(col)
        return True
    except (ValueError, TypeError):
        return False


# Retain only numeric patient ID columns
X_train = X_train[X_train.index.map(is_numeric_col)]
X_test  = X_test[X_test.index.map(is_numeric_col)]

# Reset index to expose patient IDs as a regular column
X_train.reset_index(drop=False, inplace=True)
X_test.reset_index(drop=False, inplace=True)

# Extract ordered patient ID lists before dropping the index column
train_patient_ids = X_train["index"].astype(int).tolist()
test_patient_ids  = X_test["index"].astype(int).tolist()

# Convert expression values to float arrays
X_train = X_train.drop(columns=["index"]).values.astype(float)
X_test  = X_test.drop(columns=["index"]).values.astype(float)

# Build patient-to-label mapping from the labels CSV
labels_df["patient"] = labels_df["patient"].astype(int)
label_map = dict(zip(labels_df["patient"], labels_df["cancer"]))

# Align labels with the patient order in each split
y_train_raw = [label_map[p] for p in train_patient_ids]
y_test_raw  = [label_map[p] for p in test_patient_ids]

# Part 6 - Normalização

In [ ]:
"""
Label encoding and dataset shape validation.

Encodes string class labels into integer indices using LabelEncoder,
then prints a summary of dataset dimensions and patient ID alignment.
"""

# Encode string labels into integer class indices
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test  = le.transform(y_test_raw)

# Preserve original class names for reporting and prompt generation
class_names = list(le.classes_)

# Dataset shape and label summary
print(f"Train : {X_train.shape[0]} samples | {X_train.shape[1]} genes")
print(f"Test  : {X_test.shape[0]} samples  | {X_test.shape[1]} genes")
print(f"Classes    : {class_names}")
print(f"Train IDs  : {train_patient_ids}")
print(f"Test IDs   : {test_patient_ids}")

# Part 7 - Pre-processamento

In [ ]:
"""
Preprocessing pipeline: StandardScaler and SelectKBest.

The scaler is fitted exclusively on the training set to prevent data leakage.
SelectKBest selects the K most discriminative genes using the ANOVA F-score.
selected_genes retains the original gene names for use in prompt construction.
"""

# Fit scaler on training data only and apply to both splits

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Select the K most discriminative genes via ANOVA F-score

selector    = SelectKBest(score_func=f_classif, k=K_GENES)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_test_sel  = selector.transform(X_test_scaled)

# Retrieve original gene names corresponding to selected indices

selected_idx   = selector.get_support(indices=True)
selected_genes = [gene_names[i] for i in selected_idx]

print(f"SelectKBest : {K_GENES} genes selected from {X_train.shape[1]}")
print(f"Genes       : {selected_genes}")

# Part 8 - Model LLM

In [ ]:
"""
Load BioMedGPT-LM-7B using pure float16 precision.

Quantisation via BitsAndBytesConfig is intentionally omitted.
Only torch_dtype=float16 and device_map=auto are used to distribute
the model across available CUDA devices.
"""

# Core imports for model loading

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Model and device configuration

MODEL_NAME = "PharMolix/BioMedGPT-LM-7B"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# Load tokenizer with remote code trust enabled

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

import torch
import gc

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()

# Load model using 4-bit quantization
model_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BNB_CONFIG,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# Set inference mode
model_llm.eval()

# Check GPU memory
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3

    print(f"GPU {i}: {allocated:.2f} GB / {total:.1f} GB")

print("Model loaded:", MODEL_NAME)
print("Quantization: 4-bit NF4")


# Part 9 - Prompt Engineering 1

In [ ]:
"""
Few-shot prompt builder for BioMedGPT-LM-7B.

Uses the Llama2-Chat template with [INST] <<SYS>> tags because the model
was fine-tuned from Llama2-7B-Chat. N_SHOTS examples are split equally
between ALL and AML classes.
"""


def build_prompt(x_examples, y_examples, x_new, gene_names, n_shots):
    """
    Build a few-shot prompt in Llama2-Chat format for BioMedGPT.

    Args:
        x_examples (np.ndarray): Feature matrix of training examples.
        y_examples (np.ndarray): Encoded labels of training examples.
        x_new      (np.ndarray): Feature vector of the patient to classify.
        gene_names (list)      : Names of the selected genes.
        n_shots    (int)       : Total number of few-shot examples.

    Returns:
        str: Formatted prompt ready for the tokenizer.
    """

    # System instruction passed to the model via the SYS block

    system_msg = (
        "You are an expert oncologist specializing in leukemia diagnosis. "
        "Classify the patient as ALL (Acute Lymphoblastic Leukemia) or "
        "AML (Acute Myeloid Leukemia) based on gene expression values. "
        "Answer with only ALL or AML."
    )

    # Select an equal number of ALL and AML examples for balanced few-shot context

    all_idx  = [i for i, y in enumerate(y_examples) if y == 0][: n_shots // 2]
    aml_idx  = [i for i, y in enumerate(y_examples) if y == 1][: n_shots // 2]
    shot_idx = all_idx + aml_idx

    # Build the few-shot example block

    shots = ""
    for idx in shot_idx:
        gene_str = ", ".join(
            [f"{g}: {x_examples[idx][j]:.3f}" for j, g in enumerate(gene_names)]
        )
        label  = "ALL" if y_examples[idx] == 0 else "AML"
        shots += f"Patient gene expression: {gene_str}\nDiagnosis: {label}\n\n"

    # Format the new patient gene expression string

    new_gene_str = ", ".join(
        [f"{g}: {x_new[j]:.3f}" for j, g in enumerate(gene_names)]
    )

    # Assemble the full Llama2-Chat formatted prompt

    return (
        f"[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n"
        f"{shots}"
        f"Patient gene expression: {new_gene_str}\n"
        f"Diagnosis: [/INST]"
    )


print("build_prompt defined. Ready to run inference.")

In [ ]:
"""
BioMedGPT-LM-7B inference on test patients.

Generates ALL/AML predictions and P(AML) probabilities derived from
the logits of the first generated token, enabling AUC-ROC computation.
"""

# Inference-specific imports

from transformers import GenerationConfig
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

# Generation configuration for deterministic single-token classification

gen_config_clf = GenerationConfig(
    max_new_tokens=10,
    do_sample=False,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

# Containers for predictions and calibrated probabilities

y_pred  = []
y_proba = []

print("Running inference...\n")

for i in range(len(X_test_sel)):

    # Build the few-shot prompt for the current test patient

    prompt = build_prompt(
        X_train_sel, y_train,
        X_test_sel[i], selected_genes, N_SHOTS,
    )

    # Tokenise the prompt and move tensors to the target device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(DEVICE)

    # Run inference without gradient computation

    with torch.no_grad():
        output = model_llm.generate(
            **inputs,
            generation_config=gen_config_clf,
            output_scores=True,
            return_dict_in_generate=True,
        )

    # Decode the generated sequence and isolate the model response

    response       = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
    response_clean = response[len(prompt):].strip().upper()

    # Extract token-level logits from the first generated position

    logits     = output.scores[0][0]
    proba_soft = torch.softmax(logits, dim=-1)

    # Retrieve token IDs for ALL and AML class labels

    all_tok = tokenizer.encode("ALL", add_special_tokens=False)[0]
    aml_tok = tokenizer.encode("AML", add_special_tokens=False)[0]

    # Compute normalised P(AML) from the two competing token probabilities

    p_all    = proba_soft[all_tok].item()
    p_aml    = proba_soft[aml_tok].item()
    prob_aml = p_aml / (p_all + p_aml + 1e-9)

    # Derive hard prediction from presence of class label in decoded response

    pred = 0 if "ALL" in response_clean else 1
    y_pred.append(pred)
    y_proba.append(prob_aml)

    print(
        f"  Patient {test_patient_ids[i]:>2} | "
        f"Resp: {response_clean[:8]:<8} | "
        f"Real: {'ALL' if y_test[i] == 0 else 'AML'} | "
        f"P(AML): {prob_aml:.3f}"
    )

In [ ]:
"""
Evaluation of BioMedGPT-LM-7B classification performance.

Computes accuracy, macro F1-score, and AUC-ROC from the predictions
and calibrated probabilities generated during inference. Results are
stored in a dictionary for later comparison with baseline models.
"""

# Compute classification metrics

acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred, average="macro")
auc_roc = roc_auc_score(y_test, y_proba)

# Store results under the model name for consolidated reporting

llm_results = {
    "BioMedGPT-LM-7B": {
        "y_pred"  : y_pred,
        "y_proba" : y_proba,
        "accuracy": acc,
        "f1"      : f1,
        "auc_roc" : auc_roc,
    }
}

print(f"\nAccuracy : {acc:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc_roc:.4f}")

In [ ]:
print(f"\n{classification_report(y_test, y_pred, target_names=class_names)}")

In [ ]:
from transformers import GenerationConfig

gen_config_full = GenerationConfig(
    max_new_tokens=200,
    do_sample=False,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

In [ ]:
def build_prompt_v3(x_examples, y_examples, x_new, gene_names, n_shots):
    """
    Build a few-shot prompt using a plain completion format for leukemia classification.

    Uses the 'Answer: This patient has' continuation style, which the model
    responds to reliably. Examples are balanced between ALL and AML classes
    to guide classification of the new patient.

    Args:
        x_examples (np.ndarray): Feature matrix of training examples.
        y_examples (np.ndarray): Encoded labels of training examples.
        x_new      (np.ndarray): Feature vector of the patient to classify.
        gene_names (list)      : Names of the selected genes.
        n_shots    (int)       : Total number of few-shot examples.

    Returns:
        str: Formatted prompt with few-shot examples and the new patient entry.
    """

    # Select balanced indices from ALL and AML classes

    all_idx  = [i for i, y in enumerate(y_examples) if y == 0][: n_shots // 2]
    aml_idx  = [i for i, y in enumerate(y_examples) if y == 1][: n_shots // 2]
    shot_idx = all_idx + aml_idx

    # Build each few-shot example block with gene values and a gold answer

    shots = ""
    for idx in shot_idx:
        genes_ex = "\n".join(
            [f"  {g}: {x_examples[idx][j]:.4f}" for j, g in enumerate(gene_names)]
        )

        # Construct class-specific justification referencing discriminative genes

        if y_examples[idx] == 0:
            answer = (
                "Acute Lymphoblastic Leukemia (ALL), because the high expression "
                f"of {gene_names[0]} and {gene_names[1]} are consistent with ALL markers."
            )
        else:
            answer = (
                "Acute Myeloid Leukemia (AML), because the elevated expression "
                f"of {gene_names[2]} and {gene_names[3]} are consistent with AML markers."
            )

        shots += (
            f"Gene expression values:\n{genes_ex}\n\n"
            f"Question: Is this patient ALL or AML? "
            f"Mention at least 3 genes to justify.\n\n"
            f"Answer: This patient has {answer}\n\n"
            f"---\n\n"
        )

    # Format the new patient gene expression block

    genes_new = "\n".join(
        [f"  {g}: {x_new[j]:.4f}" for j, g in enumerate(gene_names)]
    )

    # Assemble the full prompt ending with the open completion anchor

    return (
        f"Task: Classify leukemia type based on gene expression values.\n\n"
        f"{shots}"
        f"Gene expression values:\n{genes_new}\n\n"
        f"Question: Is this patient ALL or AML? "
        f"Mention at least 3 genes to justify.\n\n"
        f"Answer: This patient has"
    )

In [ ]:
"""
Leave-one-out inference on the training set using BioMedGPT-LM-7B.

Each patient is classified using the remaining training patients as
few-shot examples. Predictions, probabilities, and LLM explanations
are saved incrementally to CSV after every iteration.
"""

# Clear GPU memory before starting the inference loop

torch.cuda.empty_cache()

results   = []
save_path = "/content/resultado_llm_treino.csv"

print(f"Running inference for all {len(X_train_sel)} training patients...\n")

for i in range(len(X_train_sel)):
    try:
        # Exclude the current patient from the few-shot example pool

        idx_outros = [j for j in range(len(X_train_sel)) if j != i]
        x_examples = X_train_sel[idx_outros]
        y_examples = y_train[idx_outros]

        # Build the few-shot prompt for the current patient

        prompt = build_prompt_v3(
            x_examples, y_examples,
            X_train_sel[i], selected_genes, N_SHOTS,
        )

        # Tokenise and move inputs to the target device

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(DEVICE)

        # Run inference without gradient computation

        with torch.no_grad():
            output = model_llm.generate(
                **inputs,
                generation_config=gen_config_full,
                output_scores=True,
                return_dict_in_generate=True,
            )

        # Decode the generated sequence and isolate the new response text

        resposta_bruta = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
        resposta_nova  = resposta_bruta[len(prompt):].strip()

        # Extract hard classification label from the decoded response

        classification = extract_classification(resposta_nova)

        # Compute softmax probabilities from the first generated token logits

        logits     = output.scores[0][0]
        proba_soft = torch.softmax(logits, dim=-1)

        # Retrieve token IDs for ALL and AML surface forms and synonyms

        all_tokens = tokenizer.encode("ALL",     add_special_tokens=False)
        aml_tokens = tokenizer.encode("AML",     add_special_tokens=False)
        lym_tokens = tokenizer.encode("Lymph",   add_special_tokens=False)
        mye_tokens = tokenizer.encode("Myeloid", add_special_tokens=False)

        # Aggregate probabilities across synonym tokens for each class

        p_all    = sum(proba_soft[t].item() for t in all_tokens + lym_tokens)
        p_aml    = sum(proba_soft[t].item() for t in aml_tokens + mye_tokens)
        prob_aml = p_aml / (p_all + p_aml + 1e-9)

        # Fall back to probability-based decision when classification is ambiguous

        if classification == "UNKNOWN":
            classification = "AML" if prob_aml > 0.5 else "ALL"

        # Compute derived metrics for reporting

        real_label     = "ALL" if y_train[i] == 0 else "AML"
        confidence     = max(prob_aml, 1 - prob_aml)
        correct        = classification == real_label
        llm_explicacao = f"This patient has {resposta_nova}"

        # Append structured result record for the current patient

        results.append({
            "id"              : int(i),
            "patient_id"      : train_patient_ids[i],
            "conjunto"        : "treino",
            "real_label"      : real_label,
            "prediction"      : classification,
            "correct"         : correct,
            "probability_AML" : round(prob_aml, 4),
            "confidence"      : round(confidence, 4),
            "LLM_Explicacao"  : llm_explicacao,
            "words"           : len(llm_explicacao.split()),
        })

        status = "OK" if correct else "FAIL"
        print(
            f"  [{status}] Patient {train_patient_ids[i]:>2} | "
            f"Pred: {classification:<3} | Real: {real_label:<3} | "
            f"Conf: {confidence:.3f} | {resposta_nova[:70]}"
        )

        # Persist results incrementally after each patient

        pd.DataFrame(results).to_csv(save_path, index=False)

        # Release GPU cache every 10 patients to prevent memory exhaustion

        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            print(f"\n  GPU cache cleared ({i + 1}/{len(X_train_sel)})\n")

    except Exception as e:

        # Record error entry so the loop continues without interruption

        print(f"  Error on patient {i}: {e}")
        results.append({
            "id"              : int(i),
            "patient_id"      : train_patient_ids[i],
            "conjunto"        : "treino",
            "real_label"      : "ALL" if y_train[i] == 0 else "AML",
            "prediction"      : "ERROR",
            "correct"         : False,
            "probability_AML" : None,
            "confidence"      : None,
            "LLM_Explicacao"  : str(e),
            "words"           : 0,
        })

In [ ]:
"""
Consolidate inference results and persist the final training set predictions.

Casts column types for consistency, computes overall accuracy,
saves the DataFrame to CSV, and prints a summary report.
"""

# Build the final results DataFrame and enforce column types
df_final = pd.DataFrame(results)
df_final["id"] = df_final["id"].astype(int)
df_final["prediction"] = df_final["prediction"].astype(str)

# Compute overall accuracy across all training patients
acc_final = df_final["correct"].mean()

# Persist the consolidated results to disk
df_final.to_csv(save_path, index=False)

# Print summary report
print(f"\nSaved to        : {save_path}")
print(f"Total patients  : {len(df_final)}")
print(f"Correct         : {df_final['correct'].sum()} / {len(df_final)}\n")

# Display selected columns for a quick result overview
df_final[["patient_id", "real_label", "prediction", "correct", "confidence", "LLM_Explicacao"]]

# Preview the first six rows of the final DataFrame
df_final.head(n=6)

# Part 10 - Prompt Engineering 2

In [ ]:
""" New prompt with explanation in Portuguese for the LLM_Explanation column.

The model should generate for each patient:

- ALL or AML classification

- Names of the genes mentioned
- Expression level of each gene (high/low)

- Clinical justification in Portuguese

Runs on the 38 training patients with leave-one-out few-shot.

"""

In [ ]:
"""
Compact zero-shot prompt (v4) designed to fit within 1024 tokens.

Issues identified in prior diagnostics:
  - Prompts with 1433 tokens were truncated to 1024, cutting the question.
  - The model saw only examples and attempted to complete gene values.
  - Top predicted tokens were digits (0-9) instead of ALL or AML.

Solution:
  - Zero-shot format to eliminate few-shot overhead and stay within context.
  - Only the 5 most discriminative genes are included in the prompt.
  - Short, direct instruction written in English.
  - Response anchored with 'This patient has', a continuation the model
    already recognises from pre-training.
"""


def build_prompt_v4(x_new, gene_names):
    """
    Build a compact zero-shot prompt that fits within 1024 tokens.

    Selects only the 5 genes with the highest absolute expression values
    to keep the prompt short and ensure the question is never truncated
    by the tokenizer.

    Args:
        x_new      (np.ndarray): Feature vector of the patient to classify.
        gene_names (list)      : Names of the selected genes.

    Returns:
        str: Short formatted prompt ready for BioMedGPT inference.
    """

    # Select the 5 genes with the highest absolute expression magnitude

    top5_idx = np.argsort(np.abs(x_new))[-5:][::-1]

    # Format each selected gene with its expression value and direction label

    genes_str = "\n".join(
        [
            f"  {gene_names[j]}: {x_new[j]:.4f} "
            f"({'high' if x_new[j] > 0 else 'low'} expression)"
            for j in top5_idx
        ]
    )

    # Assemble the zero-shot prompt with an open completion anchor

    return (
        f"You are an expert oncologist specializing in leukemia.\n\n"
        f"Top gene expression values for this patient:\n{genes_str}\n\n"
        f"Classify as ALL (Acute Lymphoblastic Leukemia) or "
        f"AML (Acute Myeloid Leukemia).\n"
        f"Cite the genes above and explain the expression levels.\n\n"
        f"Answer: This patient has"
    )


# Validate token count for the first training patient against the context limit

prompt_v4_teste = build_prompt_v4(X_train_sel[0], selected_genes)
tokens_v4       = len(tokenizer(prompt_v4_teste)["input_ids"])

print(f"Tokens prompt v4 : {tokens_v4} (limit: 1024)")
print(f"Prompt:\n{prompt_v4_teste}\n")

In [ ]:
"""
Prompt v5 with balanced HIGH and LOW genes for BioMedGPT-LM-7B.

Selects the 3 genes with highest expression and 3 with lowest
to give the model real context for differentiating ALL vs AML.
Explicit instruction to cite ONLY the listed genes
and respond in Portuguese (pt-BR).
"""

import numpy as np

def build_prompt_v5(x_new, gene_names):
    """
    Builds a prompt with balanced high and low expression genes.

    Selects the 3 genes with highest expression and 3 with lowest
    to give the model real differentiation context for ALL vs AML.
    Includes instruction to generate response in Portuguese.

    Args:
        x_new      (np.ndarray): Features of the patient to classify.
        gene_names (list)      : Names of the selected genes.

    Returns:
        str: Compact prompt with balanced genes.
    """
    sorted_idx = np.argsort(x_new)
    top_high   = sorted_idx[-3:][::-1]
    top_low    = sorted_idx[:3]

    genes_high = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (HIGH)" for j in top_high]
    )
    genes_low  = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (LOW)" for j in top_low]
    )

    return (
        f"You are an expert oncologist specializing in leukemia classification.\n\n"
        f"Patient gene expression profile:\n\n"
        f"Highly expressed genes (upregulated):\n{genes_high}\n\n"
        f"Lowly expressed genes (downregulated):\n{genes_low}\n\n"
        f"ALL shows upregulation of lymphoid markers.\n"
        f"AML shows upregulation of myeloid markers.\n\n"
        f"Task: Classify this patient as ALL or AML. "
        f"Explain citing the gene names above, their expression levels, "
        f"and the clinical justification. "
        f"Write the explanation in Portuguese (pt-BR).\n\n"
        f"Answer: This patient has"
    )


def build_prompt_v6(x_new, gene_names):
    """
    Prompt v6 that forces the model to cite only the real dataset genes.

    Adds explicit instruction 'ONLY use the gene names listed above'
    to prevent hallucination of genes like LAMP1, CD63 that do not exist
    in the Golub dataset. Lists the genes twice to reinforce the constraint.

    Args:
        x_new      (np.ndarray): Features of the patient to classify.
        gene_names (list)      : Names of the selected genes.

    Returns:
        str: Prompt with anti-hallucination instruction for genes.
    """
    sorted_idx = np.argsort(x_new)
    top_high   = sorted_idx[-3:][::-1]
    top_low    = sorted_idx[:3]

    genes_high = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (HIGH)" for j in top_high]
    )
    genes_low  = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (LOW)" for j in top_low]
    )

    all_genes_listed = ", ".join(gene_names)

    return (
        f"You are an expert oncologist specializing in leukemia classification.\n\n"
        f"Available genes for this patient: {all_genes_listed}\n\n"
        f"Highly expressed genes (upregulated):\n{genes_high}\n\n"
        f"Lowly expressed genes (downregulated):\n{genes_low}\n\n"
        f"IMPORTANT: ONLY use the gene names listed above in your explanation.\n"
        f"Do NOT mention any other gene names.\n\n"
        f"ALL shows upregulation of lymphoid markers.\n"
        f"AML shows upregulation of myeloid markers.\n\n"
        f"Classify as ALL or AML and explain in Portuguese (pt-BR) "
        f"citing ONLY the genes listed above with their expression levels.\n\n"
        f"Answer: This patient has"
    )


def extract_label_from_text(text):
    """
    Extracts the classification directly from the model-generated text.

    More reliable than logits for BioMedGPT since the model
    always generates clear text mentioning ALL or AML explicitly.

    Args:
        text (str): Text generated by the model after the prompt.

    Returns:
        str: 'ALL', 'AML' or 'UNKNOWN'.
    """
    text_up = text.upper()
    if any(k in text_up for k in ["MYELOID", "AML", "ACUTE M"]):
        return "AML"
    if any(k in text_up for k in ["LYMPHOBLASTIC", "ALL", "ACUTE L", "B-CELL", "LYMPHO"]):
        return "ALL"
    return "UNKNOWN"


print("Functions defined:")
print("  build_prompt_v5")
print("  build_prompt_v6")
print("  extract_label_from_text")

In [ ]:
"""
Logit-based classification extractor and anti-hallucination prompt (v6).

extract_classification_from_logits derives the prediction directly from
the first generated token probabilities, avoiding text parsing errors.
build_prompt_v6 explicitly constrains the model to cite only genes present
in the Golub dataset, preventing hallucinated gene names such as LAMP1 and CD63.
"""


def extract_classification_from_logits(scores, tokenizer):
    """
    Extract classification directly from the logits of the first generated token.

    More reliable than text parsing because it uses the raw model probability
    before any decoding step or text hallucination can occur.

    Args:
        scores    (tuple): Score tensors returned by model.generate().
        tokenizer        : Tokenizer associated with the loaded model.

    Returns:
        tuple: (classification, prob_aml)
            - classification (str)  : Predicted class, either 'ALL' or 'AML'.
            - prob_aml       (float): Normalised probability of AML.
    """

    # Extract logits from the first generated token position

    logits     = scores[0][0]
    proba_soft = torch.softmax(logits, dim=-1)

    # Retrieve token IDs for ALL and AML surface forms and their synonyms

    all_tokens = tokenizer.encode("ALL",     add_special_tokens=False)
    aml_tokens = tokenizer.encode("AML",     add_special_tokens=False)
    lym_tokens = tokenizer.encode("Lymph",   add_special_tokens=False)
    mye_tokens = tokenizer.encode("Myeloid", add_special_tokens=False)
    ac_tokens  = tokenizer.encode("Ac",      add_special_tokens=False)
    a_tokens   = tokenizer.encode("a",       add_special_tokens=False)

    # Aggregate probabilities across all synonym tokens for each class

    p_all = sum(proba_soft[t].item() for t in all_tokens + lym_tokens)
    p_aml = sum(proba_soft[t].item() for t in aml_tokens + mye_tokens + ac_tokens)

    # Normalise and apply threshold to produce the hard prediction

    prob_aml       = p_aml / (p_all + p_aml + 1e-9)
    classification = "AML" if prob_aml > 0.5 else "ALL"

    return classification, prob_aml


def build_prompt_v6(x_new, gene_names):
    """
    Build a prompt that forces the model to cite only genes present in the dataset.

    Adds an explicit 'ONLY use the gene names listed above' instruction to
    prevent hallucination of genes such as LAMP1 and CD63 that do not exist
    in the Golub dataset. Gene names are listed twice to reinforce the constraint.

    Args:
        x_new      (np.ndarray): Feature vector of the patient to classify.
        gene_names (list)      : Names of the selected genes.

    Returns:
        str: Anti-hallucination prompt with upregulated and downregulated gene blocks.
    """

    # Identify the three most upregulated and three most downregulated genes

    sorted_idx = np.argsort(x_new)
    top_high   = sorted_idx[-3:][::-1]
    top_low    = sorted_idx[:3]

    # Format high and low expression gene blocks

    genes_high = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (HIGH)" for j in top_high]
    )
    genes_low = "\n".join(
        [f"  {gene_names[j]}: {x_new[j]:.4f} (LOW)" for j in top_low]
    )

    # List all available gene names to anchor the model's vocabulary

    all_genes_listed = ", ".join(gene_names)

    # Assemble the prompt with explicit gene constraint and completion anchor

    return (
        f"You are an expert oncologist specializing in leukemia classification.\n\n"
        f"Available genes for this patient: {all_genes_listed}\n\n"
        f"Highly expressed genes (upregulated):\n{genes_high}\n\n"
        f"Lowly expressed genes (downregulated):\n{genes_low}\n\n"
        f"IMPORTANT: ONLY use the gene names listed above in your explanation.\n"
        f"Do NOT mention any other gene names.\n\n"
        f"ALL shows upregulation of lymphoid markers.\n"
        f"AML shows upregulation of myeloid markers.\n\n"
        f"Classify as ALL or AML and explain in Portuguese (pt-BR) "
        f"citing ONLY the genes listed above with their expression levels.\n\n"
        f"Answer: This patient has"
    )


# Validate token count for the first training patient against the context limit

prompt_v5_teste = build_prompt_v5(X_train_sel[0], selected_genes)
tokens_v5       = len(tokenizer(prompt_v5_teste)["input_ids"])

print(f"Tokens prompt v5 : {tokens_v5} (limit: 1024)")
print(f"\nPrompt:\n{prompt_v5_teste}\n")

# Tokenise and move inputs to the target device

inputs = tokenizer(
    prompt_v5_teste,
    return_tensors="pt",
    truncation=True,
    max_length=1024,
).to(DEVICE)

# Run inference without gradient computation

with torch.no_grad():
    output = model_llm.generate(
        **inputs,
        generation_config=gen_config_full,
        output_scores=True,
        return_dict_in_generate=True,
    )

# Decode the generated sequence and isolate the new response text

resposta_bruta = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
resposta_nova  = resposta_bruta[len(prompt_v5_teste):].strip()

print(f"RESPONSE:")
print("=" * 60)
print(resposta_nova)
print("=" * 60)

# Compute softmax probabilities and retrieve the top 5 predicted tokens

logits               = output.scores[0][0]
proba_soft           = torch.softmax(logits, dim=-1)
top5_vals, top5_idx_tok = torch.topk(proba_soft, 5)

# Display the top 5 tokens with their decoded string and probability

print("\nTOP 5 TOKENS:")
for val, idx in zip(top5_vals, top5_idx_tok):
    token_str = tokenizer.decode([idx.item()])
    print(f"  token: {repr(token_str):<20} | prob: {val.item():.4f}")

In [ ]:
"""
Zero-shot inference loop on the training set using build_prompt_v6.

Each patient is classified independently with no few-shot examples.
Classification is derived from logits via extract_classification_from_logits.
Results are saved incrementally to CSV after every iteration to prevent
data loss on long-running Kaggle sessions.
"""

# Clear GPU memory before starting the inference loop

torch.cuda.empty_cache()

results   = []
save_path = "/content/resultado_llm_treino.csv"

print(f"Running inference for all {len(X_train_sel)} training patients...\n")

for i in range(len(X_train_sel)):
    try:
        # Build the zero-shot anti-hallucination prompt for the current patient

        prompt = build_prompt_v6(X_train_sel[i], selected_genes)

        # Tokenise and move inputs to the target device

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
        ).to(DEVICE)

        # Run inference without gradient computation

        with torch.no_grad():
            output = model_llm.generate(
                **inputs,
                generation_config=gen_config_full,
                output_scores=True,
                return_dict_in_generate=True,
            )

        # Decode the generated sequence and isolate the new response text

        resposta_bruta = tokenizer.decode(output.sequences[0], skip_special_tokens=True)
        resposta_nova  = resposta_bruta[len(prompt):].strip()

        # Derive hard classification and P(AML) directly from first-token logits

        classification, prob_aml = extract_classification_from_logits(
            output.scores, tokenizer
        )

        # Compute derived metrics for reporting

        real_label = "ALL" if y_train[i] == 0 else "AML"
        confidence = max(prob_aml, 1 - prob_aml)
        correct    = classification == real_label

        # Assemble the explanation string with a fallback for empty responses

        llm_explicacao = (
            f"Classification: {classification}. "
            f"{resposta_nova if resposta_nova else 'Model did not generate an explanation.'}"
        )

        # Append structured result record for the current patient

        results.append({
            "id"              : int(i),
            "patient_id"      : train_patient_ids[i],
            "conjunto"        : "treino",
            "real_label"      : real_label,
            "prediction"      : classification,
            "correct"         : correct,
            "probability_AML" : round(prob_aml, 4),
            "confidence"      : round(confidence, 4),
            "LLM_Explicacao"  : llm_explicacao,
            "words"           : len(llm_explicacao.split()),
        })

        status = "OK" if correct else "FAIL"
        print(
            f"  [{status}] Patient {train_patient_ids[i]:>2} | "
            f"Pred: {classification:<3} | Real: {real_label:<3} | "
            f"Conf: {confidence:.3f}\n"
            f"  LLM: {llm_explicacao[:120]}\n"
        )

        # Persist results incrementally after each patient

        pd.DataFrame(results).to_csv(save_path, index=False)

        # Release GPU cache every 10 patients to prevent memory exhaustion

        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            print(f"  GPU cache cleared ({i + 1}/{len(X_train_sel)})\n")

    except Exception as e:

        # Record error entry so the loop continues without interruption

        print(f"  Error on patient {i}: {e}")
        results.append({
            "id"              : int(i),
            "patient_id"      : train_patient_ids[i],
            "conjunto"        : "treino",
            "real_label"      : "ALL" if y_train[i] == 0 else "AML",
            "prediction"      : "ERROR",
            "correct"         : False,
            "probability_AML" : None,
            "confidence"      : None,
            "LLM_Explicacao"  : str(e),
            "words"           : 0,
        })

# Part 11 - Resultado final LLM

In [ ]:
"""
Consolidate training set inference results and persist to disk.

Casts column types for consistency, computes overall accuracy,
saves the DataFrame to CSV, and prints a summary report.
"""

# Build the final training results DataFrame and enforce column types

df_treino               = pd.DataFrame(results)
df_treino["id"]         = df_treino["id"].astype(int)
df_treino["prediction"] = df_treino["prediction"].astype(str)

# Compute overall accuracy across all training patients

acc_final = df_treino["correct"].mean()

# Persist the consolidated results to disk

df_treino.to_csv(save_path, index=False)

# Print summary report

print(f"\nSaved to        : {save_path}")
print(f"Total patients  : {len(df_treino)}")
print(f"Final accuracy  : {acc_final:.2%}")
print(f"Correct         : {df_treino['correct'].sum()} / {len(df_treino)}\n")

In [ ]:
"""
Display selected columns and a preview of the training results DataFrame.
"""

# Show relevant columns for a structured result overview

df_treino[[
    "patient_id", "real_label", "prediction",
    "correct", "confidence", "LLM_Explicacao"
]]

# Preview the first five rows of the training results DataFrame

df_treino.head()

In [ ]:
"""
Bar chart visualisation of AML probability and confidence per training patient.

Two side-by-side bar charts display probability_AML and confidence for each
patient, colour-coded by true class label. The figure is saved to disk and
rendered inline.
"""

# Sort patients by ID for consistent left-to-right ordering

df_treino = df_treino.sort_values("patient_id").reset_index(drop=True)

# Assign bar colours based on the true class label

colors = ["#2196F3" if label == "ALL" else "#FF5722" for label in df_treino["real_label"]]

# Build x-axis positions and patient ID tick labels

x      = np.arange(len(df_treino))
labels = [str(pid) for pid in df_treino["patient_id"]]

# Initialise the figure with two side-by-side subplots

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    "AML Probability and Confidence per Patient\nBioMedGPT-LM-7B | Golub et al. (1999)",
    fontsize=14,
    fontweight="bold",
)

# Left panel: P(AML) per patient with decision threshold reference line

axes[0].bar(
    x, df_treino["probability_AML"].fillna(0),
    color=colors, edgecolor="black", width=0.6,
)
axes[0].axhline(y=0.5, color="black", linestyle="--", linewidth=1, label="Threshold 0.5")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=90, fontsize=7)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Probability", fontsize=11)
axes[0].set_xlabel("Patient ID", fontsize=11)
axes[0].set_title("probability_AML per Patient", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Right panel: confidence per patient with decision threshold reference line

axes[1].bar(
    x, df_treino["confidence"].fillna(0),
    color=colors, edgecolor="black", width=0.6,
)
axes[1].axhline(y=0.5, color="black", linestyle="--", linewidth=1, label="Threshold 0.5")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=90, fontsize=7)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel("Confidence", fontsize=11)
axes[1].set_xlabel("Patient ID", fontsize=11)
axes[1].set_title("Confidence per Patient", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

# Shared legend mapping bar colours to true class labels

patch_all = mpatches.Patch(color="#2196F3", label="ALL (true label)")
patch_aml = mpatches.Patch(color="#FF5722", label="AML (true label)")
fig.legend(
    handles=[patch_all, patch_aml],
    loc="lower center",
    ncol=2,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.04),
)

plt.tight_layout()

# Save figure to disk and render inline

plt.savefig("/content/barras_prob_conf.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("Saved: barras_prob_conf.png")

In [ ]:
"""
Word cloud visualisation of LLM explanations grouped by true class label.

Stopwords are extended with generic prompt terms that carry no semantic value.
Separate word clouds are generated for ALL and AML patients using class-specific
colourmaps and saved to disk.
"""

from wordcloud import WordCloud, STOPWORDS

# Extend default stopwords with generic prompt and domain terms

extra_stopwords = {
    "patient", "gene", "expression", "This", "has", "the", "is",
    "a", "of", "and", "in", "that", "to", "with", "for", "are",
    "be", "as", "at", "an", "its", "it", "by", "from", "or",
    "on", "which", "not", "this", "have", "been", "was", "were",
    "high", "low", "highly", "lowly", "expressed", "expression",
    "values", "value", "level", "levels", "Classificacao", "AML",
    "ALL", "diagnosis", "following", "sample", "marker", "markers",
}

stopwords = STOPWORDS.union(extra_stopwords)


def build_text(df, label):
    """
    Concatenate all LLM explanations for a given class into a single string.

    Args:
        df    (pd.DataFrame): DataFrame containing LLM_Explicacao and real_label columns.
        label (str)         : Class label to filter, either 'ALL' or 'AML'.

    Returns:
        str: Single concatenated string of all explanations for the given class.
    """

    # Filter rows by class label and drop missing explanation entries

    subset = df[df["real_label"] == label]["LLM_Explicacao"].dropna()
    return " ".join(subset.tolist())


# Build concatenated explanation strings for each class

text_all = build_text(df_treino, "ALL")
text_aml = build_text(df_treino, "AML")

# Generate word cloud for ALL patients using a blue colourmap

wc_all = WordCloud(
    width=800,
    height=500,
    background_color="white",
    colormap="Blues",
    stopwords=stopwords,
    max_words=80,
    collocations=False,
).generate(text_all)

# Generate word cloud for AML patients using an orange colourmap

wc_aml = WordCloud(
    width=800,
    height=500,
    background_color="white",
    colormap="Oranges",
    stopwords=stopwords,
    max_words=80,
    collocations=False,
).generate(text_aml)

# Initialise the figure with two side-by-side subplots

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    "Word Cloud - LLM Explanations by Class\nBioMedGPT-LM-7B | Golub et al. (1999)",
    fontsize=14,
    fontweight="bold",
)

# Left panel: ALL word cloud

axes[0].imshow(wc_all, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title(
    f"ALL — {len(df_treino[df_treino['real_label'] == 'ALL'])} patients",
    fontsize=13,
    color="#1565C0",
    fontweight="bold",
)

# Right panel: AML word cloud

axes[1].imshow(wc_aml, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title(
    f"AML — {len(df_treino[df_treino['real_label'] == 'AML'])} patients",
    fontsize=13,
    color="#E65100",
    fontweight="bold",
)

plt.tight_layout()

# Save figure to disk and render inline

plt.savefig("/content/wordcloud_llm.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("Saved: wordcloud_llm.png")

In [ ]:
"""
Analytical dashboard for BioMedGPT-LM-7B training set results.

Loads the saved inference CSV and renders a 2x3 figure containing:
a probability distribution histogram, confidence boxplots, ROC curve,
probability-confidence scatter plot, explanation length boxplots,
and a summary statistics panel.
"""

from sklearn.metrics import roc_auc_score, roc_curve

# Load inference results and fill missing values with neutral defaults

df = pd.read_csv("/content/resultado_llm_treino.csv")

df["probability_AML"] = df["probability_AML"].fillna(0.5)
df["confidence"]      = df["confidence"].fillna(0.5)
df["words"]           = df["words"].fillna(0).astype(int)

# Split DataFrame by true class label for class-specific plots

df_all = df[df["real_label"] == "ALL"]
df_aml = df[df["real_label"] == "AML"]

# Prepare binary ground truth and probability arrays for ROC computation

y_true  = (df["real_label"] == "AML").astype(int).values
y_proba = df["probability_AML"].values

# Define class colours and shared legend patches

COLOR_ALL = "#2196F3"
COLOR_AML = "#FF5722"

patch_all = mpatches.Patch(color=COLOR_ALL, label="ALL (true label)", alpha=0.7)
patch_aml = mpatches.Patch(color=COLOR_AML, label="AML (true label)", alpha=0.7)

# Initialise the 2x3 dashboard figure

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    "Analytical Dashboard - BioMedGPT-LM-7B\nGolub et al. (1999) | Training Set",
    fontsize=14,
    fontweight="bold",
)

# Panel 1: histogram of P(AML) distribution per class

ax = axes[0][0]
ax.hist(
    df_all["probability_AML"], bins=12, color=COLOR_ALL,
    alpha=0.7, label="ALL", edgecolor="black",
)
ax.hist(
    df_aml["probability_AML"], bins=12, color=COLOR_AML,
    alpha=0.7, label="AML", edgecolor="black",
)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1.2, label="Threshold 0.5")
ax.set_xlabel("probability_AML", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("probability_AML Distribution", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")

# Panel 2: boxplot of confidence scores per class

ax   = axes[0][1]
bp   = ax.boxplot(
    [df_all["confidence"].values, df_aml["confidence"].values],
    patch_artist=True,
    widths=0.4,
)
bp["boxes"][0].set_facecolor(COLOR_ALL)
bp["boxes"][1].set_facecolor(COLOR_AML)
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(2)
ax.set_xticks([1, 2])
ax.set_xticklabels(["ALL", "AML"], fontsize=11)
ax.set_ylabel("Confidence", fontsize=11)
ax.set_title("Confidence Boxplot by Class", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3, axis="y")
ax.legend(handles=[patch_all, patch_aml], fontsize=9)

# Panel 3: ROC curve with AUC annotation

ax          = axes[0][2]
fpr, tpr, _ = roc_curve(y_true, y_proba)
auc_val     = roc_auc_score(y_true, y_proba)
ax.plot(fpr, tpr, color="#7B1FA2", linewidth=2.5, label=f"AUC = {auc_val:.3f}")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curve", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 4: scatter plot of P(AML) versus confidence coloured by true class

ax = axes[1][0]
ax.scatter(
    df_all["probability_AML"], df_all["confidence"],
    color=COLOR_ALL, alpha=0.7, s=80, edgecolors="black", linewidths=0.5, label="ALL",
)
ax.scatter(
    df_aml["probability_AML"], df_aml["confidence"],
    color=COLOR_AML, alpha=0.7, s=80, edgecolors="black", linewidths=0.5, label="AML",
)
ax.axvline(x=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.axhline(y=0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("probability_AML", fontsize=11)
ax.set_ylabel("Confidence", fontsize=11)
ax.set_title("Scatter probability_AML vs Confidence", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 5: boxplot of explanation word count per class with mean annotations

ax  = axes[1][1]
bp2 = ax.boxplot(
    [df_all["words"].values, df_aml["words"].values],
    patch_artist=True,
    widths=0.4,
)
bp2["boxes"][0].set_facecolor(COLOR_ALL)
bp2["boxes"][1].set_facecolor(COLOR_AML)
for median in bp2["medians"]:
    median.set_color("black")
    median.set_linewidth(2)
ax.set_xticks([1, 2])
ax.set_xticklabels(["ALL", "AML"], fontsize=11)
ax.set_ylabel("Number of Words", fontsize=11)
ax.set_title("Explanation Length by Class", fontsize=12, fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
ax.legend(handles=[patch_all, patch_aml], fontsize=9)

# Annotate each box with the class mean word count

mean_all = df_all["words"].mean()
mean_aml = df_aml["words"].mean()
ax.text(
    1, df_all["words"].max() + 2,
    f"mean: {mean_all:.1f}", ha="center", fontsize=9, color=COLOR_ALL, fontweight="bold",
)
ax.text(
    2, df_aml["words"].max() + 2,
    f"mean: {mean_aml:.1f}", ha="center", fontsize=9, color=COLOR_AML, fontweight="bold",
)

# Panel 6: summary statistics text box in the unused sixth subplot

axes[1][2].axis("off")
axes[1][2].text(
    0.5, 0.5,
    f"Summary\n\n"
    f"Total patients  : {len(df)}\n"
    f"ALL             : {len(df_all)}\n"
    f"AML             : {len(df_aml)}\n\n"
    f"Accuracy        : {df['correct'].mean():.2%}\n"
    f"AUC-ROC         : {auc_val:.4f}\n"
    f"Mean words ALL  : {mean_all:.1f}\n"
    f"Mean words AML  : {mean_aml:.1f}",
    ha="center", va="center",
    fontsize=11,
    transform=axes[1][2].transAxes,
    bbox=dict(boxstyle="round", facecolor="#F5F5F5", edgecolor="gray"),
)

plt.tight_layout()

# Save dashboard to disk and render inline

plt.savefig("/content/dashboard_llm.png",
            dpi=150, bbox_inches="tight")
plt.show()

print("Saved: dashboard_llm.png")

In [ ]:
import os
import pandas as pd

result_path = "/content/resultado_llm_treino.csv"

if os.path.exists(result_path):
    df = pd.read_csv(result_path)

    print("File hasil ditemukan!")
    print("Jumlah data:", len(df))
    print("\nKolom hasil:")
    print(df.columns.tolist())

    display(df.head())
else:
    print("File hasil belum ditemukan!")

In [ ]:
print("Jumlah prediksi:")
print(df["prediction"].value_counts())

print("\nPerbandingan hasil:")
print(pd.crosstab(
    df["real_label"],
    df["prediction"]
))

In [ ]:

import pandas as pd
import os

file_path = "/content/resultado_llm_treino.csv"

if os.path.exists(file_path):
    df_train = pd.read_csv(file_path)

    print("=== HASIL TRAINING ===")
    print("Jumlah pasien:", len(df_train))
    print("\nJumlah prediksi:")
    print(df_train["prediction"].value_counts())

    print("\nJumlah error:")
    print((df_train["prediction"] == "ERROR").sum())

    print("\nJumlah prediksi benar:")
    print(df_train["correct"].sum())

    print("\nPreview hasil:")
    display(df_train.head())
else:
    print("File CSV tidak ditemukan!")


In [ ]:

from google.colab import files
import os

# Download hasil klasifikasi
files.download("/content/resultado_llm_treino.csv")

# Download visualisasi
files.download("/content/barras_prob_conf.png")
files.download("/content/wordcloud_llm.png")
files.download("/content/dashboard_llm.png")
